# Guided Solver Config Builder — Step 1 of 2

This notebook configures all model inputs and writes a `solver_params.json` file to disk. **Run this before `guided_bayesian_inference.ipynb`.**

The worked example is run R0 of the Tier-1 plan: `a1` and `c3` on the C8 system, fitted to the Tier-1 synthetic data in `Data/Tier1_rates/Chain_C8/`. Run as is, it writes the same model, data, priors and sampler settings as `Results/Tier1/Tier1 C8 - a1c3/solver_params.json`, but into its own folder, `Results/Guided Example - C8 a1c3/`.

The four sections below cover output paths, free parameters and scaling values, experimental datasets (including the calculation module), and solver settings. Most users only need to edit **Sections 1, 2, and 3**.

*Annette Thompson · Fox and Shirts Labs, CU Boulder · 2026 · Developed with assistance from GitHub Copilot (Claude Sonnet 4.6)*

### What is `solver_params.json`?

It is a single JSON file that bundles everything the inference engine needs:
- which kinetic parameters to infer (and their prior distributions)
- the value of every other scaling group
- which experimental datasets to fit
- ODE solver tolerances and MCMC sampler settings
- per-dataset initial condition definitions (from `init_cond_columns` and/or `init_cond_overrides`)

Run the cells below in order, then execute the final cell to write the file to disk.

In [ ]:
import re
import sys
sys.path.append("../")
from pathlib import Path

from Utilities.reaction_model_builder import discover_scaling_groups
from Utilities.solver_config_builder import (
    build_solver_params_config,
    discover_observable_names,
    endpoint_dataset,
    timeseries_dataset,
    write_solver_params_json_file,
)

## Section 1: Output paths

In [ ]:
# Each run gets its own folder under Results/; change `results_folder` to start a new one.
results_folder = "Guided Example - C8 a1c3"
system = "C8"  # one rung of the chain-length ladder, Reactions/EC_FAS_ME1/<system>/

# The notebook writes the solver JSON to ../Results/<results_folder>/.
solver_params_file = "solver_params.json"

# `path_base` is the Python Model folder, stored relative to the solver JSON file
# (Results/<results_folder>/ -> ../..). Every other path inside solver_params.json is
# relative to it.
path_base = "../.."

output_paths = {
    "reactions_source": f"Reactions/EC_FAS_ME1/{system}",
    "results_save_dir": f"Results/{results_folder}",
    "prior_samples_file": "prior_samples_pm.nc",
    "posterior_samples_file": "posterior_samples_pm.nc",
    "trace_plot_file": "trace_plot.png",
}

## Section 2: Free Parameters, Priors & Scaling Values

List every model parameter you want to infer. This can be a reaction rate constant or a scaling parameter such as `a1` used in reaction-file `scaling_group` / `rvs_scaling_group` expressions. If `param_name` is a scaling parameter, every rate expression that uses it is recalculated from the sampled value during inference.

| Key | Description |
|---|---|
| `rxn_name` | Optional reaction name for documentation; scaling parameters can leave this out |
| `param_name` | Name of the parameter in the reaction-model parameter vector, e.g. a rate constant or `a1` |
| `distribution` | Prior shape — `"LogNormal"`, `"Normal"`, `"Gamma"` (any `preliz.maxent` distribution) |
| `lower` / `upper` | Interval bounds for the prior |
| `mass` | Probability mass that should fall within `[lower, upper]` |
| `fixed_stat` | Optional, e.g. `["median", 1.0]`: pins one statistic of the prior |
| `sample_scale` | Optional, `Normal` priors only: samples the parameter on `sample_scale * x`, so a narrow prior (like `d1`'s) has the same scale as the others |

The prior is the **maximum-entropy** distribution (`preliz.maxent`) with that mass in `[lower, upper]`: the least informative one consistent with your bounds. A LogNormal with a fixed median has only its width left to choose, so it is solved exactly instead. The Tier-1 priors put 95% of the mass in `[0.1, 10]` with the median at the no-op value 1.

**Scaling values.** `scaling_groups` states the value of every scaling group in the reaction files. Free parameters are sampled; the rest stay at these values. Multiplicative groups are 1 at nominal. `d`-prefixed groups enter inside an exponential and are **0** at nominal: setting one to 1 silently rescales TesA by ~4×10⁵ at C12. The inference runner refuses a config without this block.

In [ ]:
# a1 leads total production; c3 is TesA's hydrolysis multiplier. Both priors put 95%
# of their mass in [0.1, 10], with the median at the no-op value 1.
free_parameter_prior_inputs = [
    {"param_name": "a1", "distribution": "LogNormal", "lower": 0.1, "upper": 10.0,
     "mass": 0.95, "fixed_stat": ["median", 1.0]},
    {"param_name": "c3", "distribution": "LogNormal", "lower": 0.1, "upper": 10.0,
     "mass": 0.95, "fixed_stat": ["median", 1.0]},
]

# Every scaling group in the reaction files, at its nominal value.
scaling_groups = {
    "a1": 1.0, "a2": 1.0, "a3": 1.0,
    "b1": 1.0, "b2": 1.0, "b3": 1.0,
    "c1": 1.0, "c2": 1.0, "c3": 1.0, "c4": 1.0,
    "d1": 0.0, "d2": 0.0, "e": 1.0,
    "x1": 1.0, "x2": 1.0, "x3": 1.0, "x4": 1.0,
}

# Catch a group the reactions use but the dict leaves out, before the runner does.
model_base = Path("..").resolve()
missing = set(discover_scaling_groups(model_base / output_paths["reactions_source"])) - set(scaling_groups)
assert not missing, f"scaling_groups is missing {sorted(missing)}"
print(f"Free parameters: {[p['param_name'] for p in free_parameter_prior_inputs]}")
print(f"Scaling groups set: {len(scaling_groups)}")

## Section 3: Experimental Datasets + Calculation Module

Each entry points to a CSV file and tells the model how to interpret it.

Also set `calculation_module_path` in this section: the Python file that defines `OBSERVABLES`, the model outputs a dataset can be compared against. `Calculation Files/Full_FAS/FA_conc.py` provides every fatty-acid concentration (`C<n>_FA (uM)`), total fatty acid in C16 equivalents, the initial rate in µM C16/min, and mole fractions.

**Current supported dataset types:**
- `"endpoint"` — a single measurement at a fixed time (e.g., final product at t = 720 s). Set `time_values` to the measurement time(s).
- `"timeseries"` — measurements at multiple time points. Set `time_column` to the name of the time column in the CSV.

**Noise models:**
- `"relative_mean"` - sets observation noise to % (parameter = `"frac"`) of the mean absolute signal
- `"relative_pointwise"`  - sets observation noise to % (parameter = `"frac"`) of their respective values
- `"absolute"` - sets observation noise to a constant sigma value (parameter = `"value"`)
- `"relative_plus_floor"` - same as relative_pointwise but adds constant value to each sigma (parameter = `"floor"`)
- `"column"` - set observation noise to sigma column in dataset (parameter = `"column_mapping"`)
- `"groupwise"` - sets observation noise based on a group-specific variability statistic (parameter = `"statistic"`, options: "std", "sem", "mad") calculated across subsets defined by a matching column (parameter = `"group_column"`)

The Tier-1 files store, next to each value, the standard deviation its noise was drawn with (a `<column>_sigma` column), so every dataset here uses `"column"`: the likelihood uses exactly that sigma.

**`init_cond_columns`** (endpoint datasets only) — maps species names to CSV column names that hold per-row initial conditions (e.g., varying enzyme concentrations across different wells).

**`observables`** - specifies one or more calculated outputs from your simulation module to be compared against that dataset and maps them to the column in the experiment file (observable name in calculation file: observable name in experiment csv)

Initial conditions must come from each dataset description (`init_cond_columns` and/or `init_cond_overrides`); there are no global defaults in the config. Each dataset must define at least one initial condition source.

In [ ]:
# The calculation module that defines model-calculated outputs.
calculation_module_path = "Calculation Files/Full_FAS/FA_conc.py"
data_folder = f"Data/Tier1_rates/Chain_{system}"

# Discover available outputs from the calculation module and reaction network.
# `model_base` is only for notebook-side discovery; the saved config still uses `path_base` above.
available_outputs = discover_observable_names(
    calculation_module_path=calculation_module_path,
    reactions_source=output_paths["reactions_source"],
    path_base=model_base,
)

# The Tier-1 baseline: 500 uM malonyl-CoA and acetyl-CoA, 1000 uM NADPH and NADH,
# 10 uM ACP and TesA, and 1 uM of every other enzyme.
baseline = {
    "C3_MalCoA": 500.0, "C2_AcCoA": 500.0, "ACP": 10.0, "NADPH": 1000.0, "NADH": 1000.0,
    "FabD": 1.0, "FabH": 1.0, "FabG": 1.0, "FabZ": 1.0, "FabI": 1.0,
    "FabF": 1.0, "FabA": 1.0, "FabB": 1.0, "TesA": 10.0,
}
# The endpoint files give each row's initial concentrations as "<species> (uM)" columns.
per_row_initial = {species: f"{species} (uM)" for species in baseline}


def sigma_columns(outputs):
    """Each output's noise sd sits in a "<output>_sigma" column next to it."""
    return {"column_mapping": {output: f"{output}_sigma" for output in outputs}}


timeseries_outputs = ["C16 Equivalents (uM)"]
profile_outputs = [o for o in available_outputs if re.fullmatch(r"C\d+_FA(_unsat)? \(uM\)", o)]
rate_outputs = ["Initial Rate (uM C16 Equivalents/min)"]

dataset_inputs = [
    # Total fatty acid in C16 equivalents at 72, 144, ..., 720 s, at the baseline.
    timeseries_dataset(
        name=f"timeseries_Chain_{system}",
        data_file=f"{data_folder}/time_vs_conc.csv",
        output_names=timeseries_outputs,
        available_outputs=available_outputs,
        time_column="Time (s)",
        init_cond_overrides=baseline,
        noise_model="column",
        noise_params=sigma_columns(timeseries_outputs),
    ),
    # Every fatty-acid species at 720 s, at the baseline: the chain-length profile.
    endpoint_dataset(
        name=f"profile_Chain_{system}",
        data_file=f"{data_folder}/init_vs_final_conc.csv",
        output_names=profile_outputs,
        available_outputs=available_outputs,
        time_values=[720],
        init_cond_columns=per_row_initial,
        init_cond_overrides={},
        noise_model="column",
        noise_params=sigma_columns(profile_outputs),
    ),
    # Initial rates (C16 equivalents at 150 s / 150 s) under five conditions:
    # baseline, FabH 0.1 uM, FabB 0, TesA 0.5 uM, FabZ 0.
    endpoint_dataset(
        name=f"rates_Chain_{system}",
        data_file=f"{data_folder}/init_vs_rate.csv",
        output_names=rate_outputs,
        available_outputs=available_outputs,
        time_values=[150],
        init_cond_columns=per_row_initial,
        init_cond_overrides={},
        noise_model="column",
        noise_params=sigma_columns(rate_outputs),
    ),
]

print(f"Discovered {len(available_outputs)} calculated outputs.")
for dataset in dataset_inputs:
    print(f"{dataset['name']}: {list(dataset['observables'])}")

## Section 4: Sampling & Solver Settings

Controls how the posterior is sampled and how the ODE is integrated. These are the settings every Tier-1 run uses (`Notes/tier1_experiment_plan.md`, section 0).

Posterior sampling always uses **BlackJAX**, checkpointed to `<results_save_dir>/checkpoint/`. Every draw (warm-up *and* sampling) is kept, so a run can be **resumed** after a time limit, **extended** later, or have its **posterior window re-picked** afterwards with `finalize_window.py`. With `rhat_threshold` set, the run stops once it converges instead of running to `draws`.

| Setting | Notes |
|---|---|
| `draws` | Ceiling on sampling draws per chain; convergence usually ends the run first |
| `tune` | Warm-up (adaptation) steps per chain |
| `chains` | Parallel chains (vmapped on one device) |
| `target_accept` | NUTS target acceptance rate; `0.8`–`0.95` |
| `rhat_threshold` / `ess_per_split_chain` | Converged when r-hat ≤ threshold and bulk ESS ≥ 2 × chains × `ess_per_split_chain` (400 at four chains). `None` for `rhat_threshold` turns early stopping off |
| `rhat_check_every` / `convergence_consecutive_checks` | Check every this many draws; require this many passing checks in a row |
| `post_convergence_checks` | Sample this many more check intervals after converging, so LOO and other diagnostics have enough draws |
| `min_chains_for_convergence` | Chains that must remain once stranded chains (far below the best chain's log-posterior) are excluded |
| `max_total_hours` | Cap on the run's total A100-equivalent compute, summed across segments |
| `checkpoint_every_steps` | Draws between checkpoints; a killed job loses at most this many |
| `is_mass_matrix_diagonal` | Diagonal metric (default, cheap) vs a dense mass matrix |
| `rtol` / `atol` | ODE tolerances. `rtol` is chosen per system: 1e-3 up to C14+unsat, 1e-4 for C16 and C16+unsat, 1e-5 for C18 and up. The PID coefficients are the same for every system |

> **Leave `ode_solver_settings` and `ode_stepsize_controller_settings` unchanged** unless you are debugging stiff ODE issues. Loosening these tolerances can make gradients fail even if the forward solve appears faster.

To run on the cluster, submit the written config with `job_files/gpu_submit.sh` and `job_files/tier1/tier1.sbatch` (see the README).

In [ ]:
# Controls MCMC sampling and the ODE integrator: the Tier-1 production settings.
prior_sampling_settings = {"draws": 2000, "random_seed": 0}

posterior_sampling_settings = {
    "sampler": "blackjax",
    "draws": 5000,
    "tune": 300,
    "chains": 4,
    "target_accept": 0.8,
    "random_seed": 42,
    "is_mass_matrix_diagonal": True,
    "rhat_threshold": 1.01,
    "ess_per_split_chain": 50,
    "min_chains_for_convergence": 3,
    "rhat_check_every": 100,
    "convergence_consecutive_checks": 2,
    "post_convergence_checks": 1,
    "checkpoint_every_steps": 5,
    "max_total_hours": 24.0,
}

ode_solver_settings = {
    "solver_name": "Kvaerno5",
    "max_steps": 20_000,
    "dt0": 1e-6,
    "stepsize_controller": "PIDController",
}

# rtol 1e-3 is C8's per-system tolerance; atol and the PID coefficients are shared by all systems.
ode_stepsize_controller_settings = {
    "rtol": 1.0e-3,
    "atol": 1.0e-7,
    "pcoeff": 0.4,
    "icoeff": 0.3,
    "dcoeff": 0.0,
}

---
## Build and Write the Config File

Run this cell to assemble all settings into a single `solver_params.json` file and write it to disk. The output path will be printed when done.

After this cell succeeds, open **`guided_bayesian_inference.ipynb`** and set its `folder_name` to the same results folder (here `"Guided Example - C8 a1c3"`) to run and plot the fit.

In [ ]:
solver_params = build_solver_params_config(
    free_parameter_prior_inputs=free_parameter_prior_inputs,
    dataset_inputs=dataset_inputs,
    prior_sampling_settings=prior_sampling_settings,
    posterior_sampling_settings=posterior_sampling_settings,
    ode_solver_settings=ode_solver_settings,
    ode_stepsize_controller_settings=ode_stepsize_controller_settings,
    calculation_module_path=calculation_module_path,
    path_base=path_base,
    output_paths=output_paths,
    scaling_groups=scaling_groups,
)

written_solver_params_path = write_solver_params_json_file(
    solver_params_config=solver_params,
    file_directory=f"../Results/{results_folder}",
    filename=solver_params_file,
)

print(f"Wrote solver config: {written_solver_params_path}")
print(f"Results folder: {results_folder}")
print(f"Path base: {path_base}")
print(f"Reaction source: {output_paths['reactions_source']}")
print(f"Save directory: {output_paths['results_save_dir']}")
print(
    f"Configured free params: {[param['param_name'] for param in solver_params['free_kinetic_params']]}"
)
print(
    f"Configured datasets: {[dataset['name'] for dataset in solver_params['datasets']]}"
)